In [1]:
import sys
import optuna
import numpy as np
import pandas as pd

sys.path.append("..")
from feature_engineering import time_based_train_test_split, get_return, round_to_step, get_mask
from databricks_connector import get_table

pd.set_option('display.max_columns', None)
target_col = "btts"
odds_col: str = "goalNoGoal_quote_currentGG"


# features = [
#  'goalNoGoal_chance_goal',
#  'goalNoGoal_chance_goalHome',
#  'goalNoGoal_chance_goalAway',
#  'goalNoGoal_multigoal_m13',
#  'goalNoGoal_multigoal_m14',
#  'goalNoGoal_multigoal_m24',
#  'goalNoGoal_multigoal_m13Home',
#  'goalNoGoal_multigoal_m13Away',
#  'goalNoGoal_multigoal_m24Home',
#  'goalNoGoal_multigoal_m24Away',
#  'goalNoGoal_quote_realGG',
#  'goalNoGoal_quote_initialGG',
#  'goalNoGoal_quote_initialNG',
#  'goalNoGoal_quote_currentGG',
#  'goalNoGoal_quote_currentNG',
#  'goalNoGoal_quote_diffRealCurrGG',
#  'goalNoGoal_quote_diffRealCurrNG',
#  'goalNoGoal_quote_diffInitialCurrGG',
#  'goalNoGoal_quote_diffInitialCurrNG',
#  'goalNoGoal_comparison_affini',
#  'goalNoGoal_comparison_flashback',
#  'goalNoGoal_stats_avgGoalHome',
#  'goalNoGoal_stats_avgGoalTakenHome',
#  'goalNoGoal_stats_avgGoalAway',
#  'goalNoGoal_stats_avgGoalTakenAway',
#  'goalNoGoal_flashback_goal',
#  'goalNoGoal_flashback_m13',
#  'goalNoGoal_flashback_m24',
#  'goalNoGoal_flashback_m35',
#  'underOver_chance_over05HT',
#  'underOver_chance_over052HT',
#  'underOver_chance_over15HT',
#  'underOver_chance_over15',
#  'underOver_chance_over25',
#  'underOver_chance_over35',
#  'underOver_chance_over45',
#  'underOver_quote_realO',
#  'underOver_quote_initialU',
#  'underOver_quote_initialO',
#  'underOver_quote_currentU',
#  'underOver_quote_currentO',
#  'underOver_quote_diffRealCurrU',
#  'underOver_quote_diffRealCurrO',
#  'underOver_quote_diffInitialCurrU',
#  'underOver_quote_diffInitialCurrO',
#  'underOver_comparison_affini',
#  'underOver_comparison_flashback',
#  'underOver_flashback_under05HT',
#  'underOver_flashback_over05HT',
#  'underOver_flashback_under15',
#  'underOver_flashback_over15',
#  'underOver_flashback_under25',
#  'underOver_flashback_over25',
#  'underOver_flashback_under35',
#  'underOver_flashback_over35',
#  'evaluation_valScala',
#  'evaluation_valMetrica',
#  'chance1x2_quote_current1',
#  'chance1x2_quote_current2'
#  ]


features = ["underOver_chance_over15HT",
"chance1x2_quote_current1",
"chance1x2_quote_current2",
"goalNoGoal_quote_currentGG",
"goalNoGoal_chance_goal",
"goalNoGoal_flashback_goal",
"goalNoGoal_stats_avgGoalHome",
"goalNoGoal_stats_avgGoalTakenAway",
"goalNoGoal_stats_avgGoalTakenHome",
"goalNoGoal_stats_avgGoalAway"]

In [ ]:
# TODO: far scegliere a Optuna se tenere o meno una variabile
# TODO: aggiungere early stopping
# TODO: aggiungere mean e std col dato aggregato settimanalmente
# TODO: aggiungere handling dei nan
# TODO: in alcuni casi ho che il valore minimo consigliato è uguale a al massimo, questo porta a maschere che annullano il df, capire come risolvere (forse considerare metriche sui df binnati)

In [3]:
# Load data
query = f"""
            SELECT *
            FROM bet_master_analytics.strategies.{target_col}_table
        """

df_loaded = get_table(query)
df_loaded = df_loaded.sort_values('time').reset_index(drop=True)

# Train/test split
df = df_loaded.copy()
df["return"] = df.apply(lambda x: get_return(strategy=x["btts"], odds=x["goalNoGoal_quote_currentGG"]), axis=1)

df_train, df_test = time_based_train_test_split(df=df, time_col="time", train_frac=0.8)

In [28]:
# Define features and relative step
feature_bins_map = {
    "underOver_chance_over15HT": 5, # 5,
    # "chance1x2_quote_current1": 1, #0.5, #0.1,
    # "chance1x2_quote_current2": 1, #0.5, #0.1,
    "goalNoGoal_chance_goal": 5, # ,
    # "goalNoGoal_flashback_goal": 10, # 5,
    # "goalNoGoal_stats_avgGoalHome": 1, #0.1, 
    # "goalNoGoal_stats_avgGoalTakenAway": 1, #0.1, 
    # "goalNoGoal_stats_avgGoalTakenHome": 1, #0.1, 
    # "goalNoGoal_stats_avgGoalAway": 1, #0.1, 
    # "goalNoGoal_quote_currentGG": 1, #0.5, #0.1
 }

# Create the binned dataframe
df_binned = df_train.copy()

for feat, step in feature_bins_map.items():
    df_binned[feat] = [round_to_step(x, step) for x in df_binned[feat]]

    if isinstance(step, int):
        df_binned[feat] = df_binned[feat].astype("Int64")


for feat in feature_bins_map.keys():
    print(feat +"\n")
    print(df_binned[feat].sort_values(ascending=True).unique())
    print("\n")

underOver_chance_over15HT

<IntegerArray>
[15, 20, 25, 30, 35, 40, 45, 50, 55, 60, 65, 70]
Length: 12, dtype: Int64


goalNoGoal_chance_goal

<IntegerArray>
[30, 35, 40, 45, 50, 55, 60, 65, 70]
Length: 9, dtype: Int64




In [ ]:
# Define the objective function
target_col = "return"
min_obs = 30
low_cardinality_threshold = 3  # <= 10 valori unici => tratto come discreta ordinata


def objective(trial):
    mask = pd.Series(True, index=df_binned.index)

    for feat, step in feature_bins_map.items():
        s = df_binned[feat]

        # salta feature non numeriche
        if not pd.api.types.is_numeric_dtype(s):
            continue

        non_null = s.dropna()
        if non_null.empty:
            continue

        nunique = non_null.nunique()

        # se vuoi escludere sempre i missing
        include_missing = False
        include_missing = trial.suggest_categorical(
            f"{feat}_include_missing",
            [True, False]
            )
     

        # ---------------------------------
        # CASO 1: numerica discreta ordinata
        # ---------------------------------
        if nunique <= low_cardinality_threshold:
            unique_vals = sorted(non_null.unique().tolist())

            min_idx = trial.suggest_int(f"{feat}_min_idx", 0, len(unique_vals) - 1, step=step)
            max_idx = trial.suggest_int(f"{feat}_max_idx", min_idx, len(unique_vals) - 1, step=step)

            feat_min = unique_vals[min_idx]
            feat_max = unique_vals[max_idx]

            feat_mask = s.between(feat_min, feat_max)

            if include_missing:
                feat_mask = feat_mask | s.isna()
            else:
                feat_mask = feat_mask & s.notna()

        # ---------------------------------
        # CASO 2: numerica continua/intera
        # ---------------------------------
        else:
            if pd.api.types.is_integer_dtype(s):
                feat_min = trial.suggest_int(
                    f"{feat}_min",
                    int(non_null.min()),
                    int(non_null.max()),
                    step=step
                )
                feat_max = trial.suggest_int(
                    f"{feat}_max",
                    feat_min,
                    int(non_null.max()),
                    step=step
                )
            else:
                feat_min = trial.suggest_float(
                    f"{feat}_min",
                    float(non_null.min()),
                    float(non_null.max()),
                    step=step
                )
                feat_max = trial.suggest_float(
                    f"{feat}_max",
                    feat_min,
                    float(non_null.max()),
                    step=step
                )

            feat_mask = s.between(feat_min, feat_max)

            if include_missing:
                feat_mask = feat_mask | s.isna()
            else:
                feat_mask = feat_mask & s.notna()

        mask &= feat_mask

    selected = df_binned.loc[mask]

    if len(selected) < min_obs:
        return -1e9

    mean_return = selected[target_col].mean()

    score = mean_return * np.sqrt(len(selected))
    return float(score)


study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=500)

print("Best params:", study.best_params)
print("Best score:", study.best_value)

[I 2026-03-27 17:24:23,401] A new study created in memory with name: no-name-62bb1c9c-f26e-46c7-9e23-0f8cf6304b08
[I 2026-03-27 17:24:23,409] Trial 0 finished with value: -0.72663940113175 and parameters: {'underOver_chance_over15HT_include_missing': False, 'underOver_chance_over15HT_min': 20, 'underOver_chance_over15HT_max': 40, 'goalNoGoal_chance_goal_include_missing': True, 'goalNoGoal_chance_goal_min': 65, 'goalNoGoal_chance_goal_max': 70}. Best is trial 0 with value: -0.72663940113175.
[I 2026-03-27 17:24:23,414] Trial 1 finished with value: -0.779487010796203 and parameters: {'underOver_chance_over15HT_include_missing': True, 'underOver_chance_over15HT_min': 25, 'underOver_chance_over15HT_max': 65, 'goalNoGoal_chance_goal_include_missing': False, 'goalNoGoal_chance_goal_min': 70, 'goalNoGoal_chance_goal_max': 70}. Best is trial 0 with value: -0.72663940113175.
[I 2026-03-27 17:24:23,418] Trial 2 finished with value: 0.3754939440333641 and parameters: {'underOver_chance_over15HT_i

Best params: {'underOver_chance_over15HT_include_missing': True, 'underOver_chance_over15HT_min': 20, 'underOver_chance_over15HT_max': 30, 'goalNoGoal_chance_goal_include_missing': True, 'goalNoGoal_chance_goal_min': 60, 'goalNoGoal_chance_goal_max': 65}
Best score: 1.117417743636239


In [30]:
study.best_params

{'underOver_chance_over15HT_min': 25,
 'underOver_chance_over15HT_max': 30,
 'goalNoGoal_chance_goal_min': 60,
 'goalNoGoal_chance_goal_max': 70}

In [31]:
# Get the resulting dataframes
params_dict =study.best_params

train_mask = get_mask(df_train, params_dict)
test_mask = get_mask(df_test, params_dict)

df_train_filtered = df_train[train_mask]
df_test_filtered = df_test[test_mask]

In [32]:
mean_roi = df_train_filtered["return"].mean()
right = df_train_filtered["btts"].sum()  
total = df_train_filtered.shape[0]

accuracy = np.round((right / total), 1)
print(f"Mean ROI: {mean_roi}")
print(f"Accuracy: {accuracy} ({right}/{total})")

Mean ROI: 0.16749999999999998
Accuracy: 0.8 (3/4)


In [33]:
mean_roi = df_test_filtered["return"].mean()
right = df_test_filtered["btts"].sum()  
total = df_test_filtered.shape[0]

accuracy = np.round((right / total), 1)
print(f"Mean ROI: {mean_roi}")
print(f"Accuracy: {accuracy} ({right}/{total})")

Mean ROI: -1.0
Accuracy: 0.0 (0/1)


In [27]:
df_train["underOver_chance_over15HT"] == 20

0       False
1       False
2       False
3       False
4       False
        ...  
4050    False
4051    False
4052    False
4053    False
4054    False
Name: underOver_chance_over15HT, Length: 4055, dtype: bool

In [25]:
study.best_params

{'underOver_chance_over15HT_min': 20,
 'underOver_chance_over15HT_max': 20,
 'goalNoGoal_chance_goal_min': 40,
 'goalNoGoal_chance_goal_max': 40}